# SHOT: YOLOv8n Tennis Ball Detector (Colab)

1. Roboflow 데이터 다운로드 (9,737장)
2. TrackNet → YOLO 변환 (19,835장)
3. 병합 후 로컬 SSD 복사
4. YOLOv8n 학습 (100에폭)
5. ONNX 변환 → Google Drive 저장

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 셀 0: 설치 + GPU 확인
!pip install ultralytics roboflow -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 셀 1: Google Drive 마운트 + Kaggle 데이터 다운로드
from google.colab import drive
drive.mount('/content/drive')

# Kaggle API 설정
import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Kaggle API 토큰 설정
import json
kaggle_token = {"username": "faultfoot", "key": "KGAT_4918b6a1d2e082177fd903adbe45271a"}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_token, f)
!chmod 600 /root/.kaggle/kaggle.json

# TrackNet 데이터 다운로드
!kaggle datasets download faultfoot/shot-tracknet-ball-data -p /content/tracknet_data --unzip -q
print("TrackNet download done")

# 확인
!ls /content/tracknet_data/

In [ ]:
# 셀 2: Roboflow 다운로드
from roboflow import Roboflow
rf = Roboflow(api_key='TdYIJQTWeBtbsT2vq0Sm')
project = rf.workspace('tennisball-3eqxr').project('tennis-ball-detection-qaxae')
dataset = project.version(1).download('yolov8', location='/content/roboflow_ball')
print("Roboflow download done")

In [ ]:
# 셀 3: TrackNet → YOLO 변환 + 병합
import json, os, shutil
from pathlib import Path
import numpy as np

# TrackNet JSON 경로 찾기
tracknet_json = None
for root, dirs, files in os.walk('/content/tracknet_data'):
    for f in files:
        if f == 'ball_combined.json':
            tracknet_json = os.path.join(root, f)
            tracknet_frames = os.path.join(root, 'frames')
            break

print(f"TrackNet JSON: {tracknet_json}")
print(f"TrackNet frames: {tracknet_frames}")

with open(tracknet_json) as f:
    annotations = json.load(f)
print(f"Total annotations: {len(annotations)}")

# === TrackNet → YOLO 변환 ===
YOLO_DIR = Path('/content/yolo_tracknet')
BALL_W, BALL_H = 0.015, 0.02

import re
from collections import defaultdict

video_frames = defaultdict(list)
for ann in annotations:
    match = re.match(r'(.+?)_frame_\d+', Path(ann['image']).stem)
    vid_id = match.group(1) if match else Path(ann['image']).stem
    video_frames[vid_id].append(ann)

video_ids = sorted(video_frames.keys())
np.random.seed(42)
np.random.shuffle(video_ids)
split_idx = max(1, int(len(video_ids) * 0.85))

splits = {'train': video_ids[:split_idx], 'val': video_ids[split_idx:]}

for split_name, vids in splits.items():
    img_dir = YOLO_DIR / split_name / 'images'
    lbl_dir = YOLO_DIR / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    
    count = 0
    for vid in vids:
        for ann in video_frames[vid]:
            src = Path(tracknet_frames) / ann['image']
            if not src.exists():
                continue
            dst = img_dir / ann['image']
            if not dst.exists():
                os.link(str(src), str(dst))
            
            lbl_path = lbl_dir / f"{Path(ann['image']).stem}.txt"
            if ann['visibility'] > 0 and ann['x'] >= 0:
                x_c = np.clip(ann['x'], 0, 1)
                y_c = np.clip(ann['y'], 0, 1)
                with open(lbl_path, 'w') as f:
                    f.write(f"0 {x_c:.6f} {y_c:.6f} {BALL_W} {BALL_H}\n")
            else:
                lbl_path.touch()
            count += 1
    print(f"TrackNet {split_name}: {count}")

# === 병합 (로컬 SSD) ===
MERGED = Path('/content/merged_ball')
for split in ['train', 'val']:
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

counts = {}

# TrackNet
for split in ['train', 'val']:
    src_img = YOLO_DIR / split / 'images'
    src_lbl = YOLO_DIR / split / 'labels'
    c = 0
    for f in src_img.glob('*'):
        dst = MERGED / split / 'images' / f'tn_{f.name}'
        if not dst.exists(): os.link(str(f), str(dst))
        lbl = src_lbl / f'{f.stem}.txt'
        dst_lbl = MERGED / split / 'labels' / f'tn_{f.stem}.txt'
        if lbl.exists() and not dst_lbl.exists(): os.link(str(lbl), str(dst_lbl))
        c += 1
    counts[f'tracknet_{split}'] = c

# Roboflow
rf_map = {'train': 'train', 'valid': 'val', 'test': 'train'}
for rf_split, my_split in rf_map.items():
    src_img = Path(f'/content/roboflow_ball/{rf_split}/images')
    src_lbl = Path(f'/content/roboflow_ball/{rf_split}/labels')
    if not src_img.exists(): continue
    c = 0
    for f in src_img.glob('*'):
        dst = MERGED / my_split / 'images' / f'rf_{f.name}'
        if not dst.exists(): os.link(str(f), str(dst))
        lbl = src_lbl / f'{f.stem}.txt'
        dst_lbl = MERGED / my_split / 'labels' / f'rf_{f.stem}.txt'
        if lbl.exists() and not dst_lbl.exists(): os.link(str(lbl), str(dst_lbl))
        c += 1
    counts[f'roboflow_{rf_split}->{my_split}'] = c

train_imgs = len(list((MERGED / 'train' / 'images').iterdir()))
val_imgs = len(list((MERGED / 'val' / 'images').iterdir()))

print(f'\n=== Merged Dataset ===')
for k, v in sorted(counts.items()):
    print(f'  {k}: {v}')
print(f'\nTotal train: {train_imgs}')
print(f'Total val: {val_imgs}')
print(f'Grand total: {train_imgs + val_imgs}')

# data.yaml
yaml_content = f"""path: {MERGED}
train: train/images
val: val/images

nc: 1
names: ['ball']
"""
(MERGED / 'data.yaml').write_text(yaml_content)
print('data.yaml written')

In [ ]:
# 셀 4: YOLOv8n 학습
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f"{MERGED}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    project='/content/models',
    name='ball_yolov8n',
    patience=15,
    save_period=10,
    lr0=0.001,
    lrf=0.01,
    optimizer='AdamW',
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    pretrained=True,
    verbose=True,
)

print("\n=== Training Complete ===")

In [ ]:
# 셀 5: ONNX 변환 + Google Drive 저장
import shutil

best_pt = '/content/models/ball_yolov8n/weights/best.pt'
model = YOLO(best_pt)

# ONNX 변환
model.export(format='onnx', imgsz=640, simplify=True, opset=18)

onnx_path = best_pt.replace('.pt', '.onnx')
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f"ONNX: {onnx_path} ({size_mb:.1f}MB)")

# Google Drive에 복사
drive_dir = '/content/drive/MyDrive/SHOT_models'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(onnx_path, f'{drive_dir}/ball_yolov8n.onnx')
shutil.copy(best_pt, f'{drive_dir}/ball_yolov8n_best.pt')

print(f"\nSaved to Google Drive: {drive_dir}")
for f in os.listdir(drive_dir):
    s = os.path.getsize(f'{drive_dir}/{f}') / 1024 / 1024
    print(f"  {f}: {s:.1f}MB")

print("\n완료! ball_yolov8n.onnx를 Android에 적용하세요.")